In [1]:
import torch
import torch.nn.functional as F
from inference import AdversarialLanguageAdapter, preprocess_single_file

/Users/hbbg/environments/tidylang/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model Loading

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Mock Language Map - Should be from TidyLang training corpus
idx2lang = {i: f"lang_{i}" for i in range(35)}

model = AdversarialLanguageAdapter(
    num_languages=len(idx2lang),
    num_train_speakers=3566,
    alpha=0.0,
    device=device
)


checkpoint_path = "best_adapter.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state'], strict=False)
print("DisentangLID Model loaded successfully.")
model.eval()


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 773/773 [00:00<00:00, 7665.84it/s]


DisentangLID Model loaded successfully.


AdversarialLanguageAdapter(
  (encoder): M3LMSpeechEncoder(
    (semantic_encoder): Wav2Vec2BertModel(
      (feature_projection): Wav2Vec2BertFeatureProjection(
        (layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
        (projection): Linear(in_features=160, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (encoder): Wav2Vec2BertEncoder(
        (dropout): Dropout(p=0.0, inplace=False)
        (layers): ModuleList(
          (0-23): 24 x Wav2Vec2BertEncoderLayer(
            (ffn1_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (ffn1): Wav2Vec2BertFeedForward(
              (intermediate_dropout): Dropout(p=0.0, inplace=False)
              (intermediate_dense): Linear(in_features=1024, out_features=4096, bias=True)
              (intermediate_act_fn): SiLU()
              (output_dense): Linear(in_features=4096, out_features=1024, bias=True)
              (output_dropout): Dropout(p=0.

# Single File Inference

In [3]:
test_audio_file = "samples/en_40527567.wav"

sem, aco, mask = preprocess_single_file(test_audio_file)

with torch.no_grad():
    lang_logits, _, _ = model(sem, aco, mask)
    probabilities = F.softmax(lang_logits, dim=-1)
    pred_idx = probabilities.argmax(dim=-1).item()
    confidence = probabilities[0, pred_idx].item()
    print(f"Predicted Language: {idx2lang.get(pred_idx, 'Unknown')}")
    print(f"Confidence Score:   {confidence:.2%}")

Predicted Language: lang_11
Confidence Score:   99.99%
